# Meteorite Baseline Training

This notebook is the main training entry point.

Design:
- notebook sets parameters, starts training and displays results
- `models/` and `tools/` implement the underlying operations
- This layout supports interactive study and longer server runs

## 1. Import dependencies

In [ ]:
from pathlib import Path
import sys

# Use the current notebook directory as the project root
PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("models folder exists:", (PROJECT_ROOT / "models").exists())
print("tools folder exists:", (PROJECT_ROOT / "tools").exists())

In [ ]:
from pathlib import Path
import json
import pandas as pd


import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Audit near duplicates
import numpy as np
from PIL import Image, ImageOps
from tqdm.auto import tqdm


from models import build_model
from tools.dataset_tools import (
    MeteoriteDataset,
    attach_md5_group_ids,
    group_aware_train_val_split,
    load_test_dataframe,
    load_train_dataframe,
    stratified_sample_dataframe,
    stratified_train_val_split,
)


from tools.submission_tools import make_submission, predict_on_test
from tools.train_tools import (
    IntegerLabelBCEWithLogitsLoss,
    build_bce_pos_weight,
    evaluate,
    set_seed,
    train_one_epoch,
)
from tools.transforms_tools import build_eval_transforms, build_train_transforms
from tools.visualize_tools import plot_label_distribution, plot_training_curves, save_sample_grid


## 2. Configure training

Adjust the settings below for each experiment.

Key settings appear in the run directory name. The full configuration is written to `run_summary.json`.

`loss_name` accepts `cross_entropy` or `bce_with_logits`.

In [ ]:
from datetime import datetime

CONFIG = {
    'data_root': PROJECT_ROOT,
    'output_base_dir': PROJECT_ROOT / 'runs',

    'model_name': 'dinov2_small',
    'pretrained': True,

    'epochs': 10,
    'batch_size': 16,
    'img_size': 224,
    'lr': 3e-4,
    'weight_decay': 1e-4,
    'val_ratio': 0.2,
    'scheduler': 'cosine',
    'loss_name': 'cross_entropy',  # Options: 'cross_entropy', 'bce_with_logits'

    'subset_size': 0,
    'test_subset_size': 0,

    'seed': 42,
    'num_workers': 4,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}



def format_float_for_name(value):
    text = f"{value:g}"
    return text.replace("+", "")


base_exp_name = '_'.join([
    f"img{CONFIG['img_size']}",
    CONFIG['model_name'],
    'adamw',
    f"lr{format_float_for_name(CONFIG['lr'])}",
    f"wd{format_float_for_name(CONFIG['weight_decay'])}",
    f"bs{CONFIG['batch_size']}",
    f"val{format_float_for_name(CONFIG['val_ratio'])}",
    f"seed{CONFIG['seed']}",
    f"sch{CONFIG['scheduler']}",
    f"loss{CONFIG['loss_name']}",
    f"pre{int(bool(CONFIG['pretrained']))}",
    f"subset{CONFIG['subset_size'] if CONFIG['subset_size'] and CONFIG['subset_size'] > 0 else 'full'}",
    f"e{CONFIG['epochs']}",
])

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

CONFIG['experiment_name'] = f"{timestamp}_{base_exp_name}"
CONFIG['output_dir'] = CONFIG['output_base_dir'] / CONFIG['experiment_name']
CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)

print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()}, ensure_ascii=False, indent=2))


## 3. Set the seed and load the data

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

dataset_dir = PROJECT_ROOT / "Dataset"

# List subdirectories
for sub in sorted(dataset_dir.iterdir()):
    if sub.is_dir():
        img_count = len(list(sub.glob("*"))) + len(list(sub.glob("*/*")))  # Approximate counts
        print(f"{sub.name}/")
        for subsub in sorted(sub.iterdir()):
            if subsub.is_dir():
                print(f"  {subsub.name}/ ({len(list(subsub.glob('*')))} images)")


In [ ]:
def count_cross_split_groups(df_train, df_val, group_col):
    """
    Count groups present in both the training and validation partitions.
    With a group-aware split on this column, the expected count is 0.
    """
    if group_col is None:
        return None
    if group_col not in df_train.columns or group_col not in df_val.columns:
        return None

    train_groups = set(df_train[group_col].dropna().astype(str).tolist())
    val_groups = set(df_val[group_col].dropna().astype(str).tolist())
    return len(train_groups.intersection(val_groups))


def drop_exact_duplicates_in_train(df, group_col='group_id'):
    """
    Remove exact duplicates within the training partition:
    Keep one image per MD5 group.
    """
    if group_col not in df.columns:
        raise ValueError(f'Column {group_col} is missing; cannot deduplicate.')

    candidate_path_cols = ['img_path', 'image_path', 'filepath', 'file_path', 'path']
    sort_cols = [col for col in candidate_path_cols if col in df.columns]

    if sort_cols:
        dedup_df = (
            df.sort_values([group_col] + sort_cols)
              .drop_duplicates(subset=group_col, keep='first')
              .reset_index(drop=True)
        )
    else:
        dedup_df = (
            df.sort_values(group_col)
              .drop_duplicates(subset=group_col, keep='first')
              .reset_index(drop=True)
        )

    removed_count = len(df) - len(dedup_df)
    return dedup_df, removed_count


def compute_dhash_int(image_path, hash_size=8):
    """
    Compute the dHash of one image as an integer.
    """
    image = Image.open(image_path)
    image = ImageOps.exif_transpose(image).convert('L')
    image = image.resize((hash_size + 1, hash_size), Image.Resampling.LANCZOS)

    pixels = np.asarray(image, dtype=np.int16)
    diff = pixels[:, 1:] > pixels[:, :-1]

    hash_value = 0
    for bit in diff.flatten():
        hash_value = (hash_value << 1) | int(bit)

    return hash_value


def hamming_distance_int(x, y):
    """
    Return the Hamming distance between integer hashes.
    pandas can promote integer hashes in columns with missing values to float, 
    Convert to Python integers before bitwise operations.
    """
    return (int(x) ^ int(y)).bit_count()


class UnionFind:
    """
    Use union-find to merge similar MD5 groups into a single near-duplicate family.
    """
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra = self.find(a)
        rb = self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            self.parent[ra] = rb
        elif self.rank[ra] > self.rank[rb]:
            self.parent[rb] = ra
        else:
            self.parent[rb] = ra
            self.rank[ra] += 1


def build_near_duplicate_families_from_md5_groups(
    df,
    path_col='img_path',
    md5_group_col='group_id',
    threshold=6,
    hash_size=8,
):
    """
    Select one image per MD5 group and use dHash to merge closely matching groups into a near-duplicate family.
    """
    if md5_group_col not in df.columns:
        raise ValueError(f'Column {md5_group_col} is missing; cannot construct near-duplicate family.')
    if path_col not in df.columns:
        raise ValueError(f'Column {path_col} is missing; cannot load images.')

    rep_df = (
        df.sort_values([md5_group_col, path_col])
          .drop_duplicates(subset=md5_group_col, keep='first')
          .reset_index(drop=True)
          .copy()
    )

    dhash_list = []
    failed_paths = []

    for image_path in tqdm(rep_df[path_col].tolist(), desc='Hash each MD5 group representative with dHash'):
        try:
            dhash_list.append(compute_dhash_int(image_path, hash_size=hash_size))
        except Exception:
            dhash_list.append(None)
            failed_paths.append(str(image_path))

    # dHash can reach the maximum unsigned 64-bit value and overflow a signed pandas Int64.
    # Store Python integers and None in an object column to prevent overflow.
    rep_df['dhash'] = pd.Series(dhash_list, dtype='object')

    valid_mask = rep_df['dhash'].notna()
    valid_rep_df = rep_df[valid_mask].reset_index(drop=True).copy()
    valid_rep_df['dhash'] = valid_rep_df['dhash'].map(int)

    n = len(valid_rep_df)
    uf = UnionFind(n)

    for i in tqdm(range(n), desc='Build near-duplicate family'):
        hi = valid_rep_df.loc[i, 'dhash']
        for j in range(i + 1, n):
            hj = valid_rep_df.loc[j, 'dhash']
            if hamming_distance_int(hi, hj) <= threshold:
                uf.union(i, j)

    root_to_family = {}
    family_ids = []

    for i in range(n):
        root = uf.find(i)
        if root not in root_to_family:
            root_to_family[root] = f'nearfam_{len(root_to_family):05d}'
        family_ids.append(root_to_family[root])

    valid_rep_df['near_dup_family_id'] = family_ids

    invalid_rep_df = rep_df[~valid_mask].copy()
    invalid_rep_df['near_dup_family_id'] = [
        f'nearfam_failed_{i:05d}' for i in range(len(invalid_rep_df))
    ]

    rep_family_df = pd.concat(
        [
            valid_rep_df[[md5_group_col, 'dhash', 'near_dup_family_id']],
            invalid_rep_df[[md5_group_col, 'dhash', 'near_dup_family_id']],
        ],
        axis=0,
        ignore_index=True,
    )


    out_df = df.merge(
        rep_family_df[[md5_group_col, 'near_dup_family_id', 'dhash']],
        on=md5_group_col,
        how='left',
    )

    return out_df, rep_family_df, failed_paths


In [ ]:
set_seed(CONFIG['seed'])
device = torch.device(CONFIG['device'])
print('device =', device)

# 1. Load only the competition training labels
full_train_df = load_train_dataframe(CONFIG['data_root'])
print('Full training sample count =', len(full_train_df))
print(f"Training class distribution:\n{full_train_df['label'].value_counts().sort_index()}")

# 2. Sample the dataset (subset_size = 0 uses all rows)
train_pool_df = stratified_sample_dataframe(
    df=full_train_df,
    label_col='label',
    sample_size=CONFIG['subset_size'],
    seed=CONFIG['seed'],
)
print('Training pool after sampling =', len(train_pool_df))

# 3. Group exact duplicates by MD5
train_pool_df = attach_md5_group_ids(train_pool_df)
md5_group_col = 'group_id'
path_col = 'img_path'

print('Unique MD5 groups =', train_pool_df[md5_group_col].nunique())

# 4. Build on MD5 groups to form near-duplicate family
near_dup_threshold = 6
near_dup_hash_method = 'dhash'

train_pool_df, near_dup_rep_df, near_dup_failed_paths = build_near_duplicate_families_from_md5_groups(
    df=train_pool_df,
    path_col=path_col,
    md5_group_col=md5_group_col,
    threshold=near_dup_threshold,
    hash_size=8,
)

print('near-duplicate family count =', train_pool_df['near_dup_family_id'].nunique())
print('dHash hashing failures =', len(near_dup_failed_paths))

# 5. Use near-duplicate families for a group-aware split (validation)
split_method = 'group_aware_neardup_dhash'
group_key = 'near_dup_family_id'

train_df, val_df = group_aware_train_val_split(
    df=train_pool_df,
    label_col='label',
    group_col=group_key,
    val_ratio=CONFIG['val_ratio'],
    seed=CONFIG['seed'],
)
print('After splitting: train =', len(train_df), 'val =', len(val_df))

# 6. Audit the split for leakage
cross_split_dup_count = count_cross_split_groups(train_df, val_df, md5_group_col)
cross_split_neardup_count = count_cross_split_groups(train_df, val_df, group_key)

print('split_method =', split_method)
print('group_key =', group_key)
print('cross_split_dup_count =', cross_split_dup_count)
print('cross_split_neardup_count =', cross_split_neardup_count)

# 7. Remove exact duplicates within the training partition
train_size_before_exact_dedup = len(train_df)
train_df, train_exact_dup_removed = drop_exact_duplicates_in_train(
    train_df,
    group_col=md5_group_col,
)
train_size_after_exact_dedup = len(train_df)

print('Training samples before exact deduplication =', train_size_before_exact_dedup)
print('Training samples after exact deduplication =', train_size_after_exact_dedup)
print('Training duplicates removed =', train_exact_dup_removed)

if cross_split_dup_count not in (None, 0):
    print(f'[WARN] md5 duplicate groups still span train/val: {cross_split_dup_count}')

if cross_split_neardup_count not in (None, 0):
    print(f'[WARN] near-duplicate family still spans train/val: {cross_split_neardup_count}')

## 4. Save split metadata and plots

In [ ]:
split_dir = CONFIG['output_dir'] / 'splits'
fig_dir = CONFIG['output_dir'] / 'figures'
split_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

train_df[['id', 'label', 'img_path']].to_csv(split_dir / 'train_split.csv', index=False)
val_df[['id', 'label', 'img_path']].to_csv(split_dir / 'val_split.csv', index=False)

plot_label_distribution(train_df, fig_dir / 'train_label_distribution.png')
plot_label_distribution(val_df, fig_dir / 'val_label_distribution.png')
print('Saved class distribution')


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from pathlib import Path
from sklearn.svm import SVC
# Import the test-data loader
from tools.dataset_tools import load_test_dataframe

# ==========================================
# 0. Load test data
# ==========================================
print("📂 Load test data...")
test_df = load_test_dataframe(CONFIG['data_root'])

# ==========================================
# 1. Configure test-time augmentations
# ==========================================
print("🚀 Initialize DINOv2 preprocessing (including TTA)...")

# Base transform (training and the unflipped test view)
base_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

# Define three test views"
tta_transforms = [
    base_transform, # View 1: original
    transforms.Compose([transforms.RandomHorizontalFlip(p=1.0), base_transform]), # View 2: horizontal flip
    transforms.Compose([transforms.RandomVerticalFlip(p=1.0), base_transform])    # View 3: vertical flip
]

# Use only the original view for training features
train_dataset = MeteoriteDataset(train_df, transforms=base_transform)
extract_train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False, num_workers=CONFIG['num_workers'])

# ==========================================
# 2. Load the locally cloned model (offline)
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"⚙️ Loading the model from /data/my-dinov2-repo...")

# ✅ Use the local Git checkout
local_repo_dir = '/data/my-dinov2-repo'

dinov2_model = torch.hub.load(
    repo_or_dir=local_repo_dir,
    model='dinov2_vitl14_reg',
    source='local'
)
dinov2_model.eval()
dinov2_model.to(device)

# ==========================================
# 3. Extract features (multi-view test inference)
# ==========================================
# ... (Reuse the extract_embeddings function above) ...

print("⏳ Extracting training embeddings (single view)...")
train_X, train_y = extract_embeddings(extract_train_loader, dinov2_model, device, is_test=False)

print("⏳ Extracting test embeddings (multi-view TTA)...")
test_X_tta_list = []

# Repeat feature extraction once per test transform
for i, t in enumerate(tta_transforms):
    print(f"   -> Extracting TTA view {i+1}/3 ...")
    # Select the current test transform
    test_dataset = MeteoriteDataset(test_df, transforms=t)
    extract_test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=CONFIG['num_workers'])
    
    # Extract embeddings for this view
    curr_test_X, test_ids = extract_embeddings(extract_test_loader, dinov2_model, device, is_test=True)
    test_X_tta_list.append(curr_test_X)

# ⭐️ Average the three feature vectors element by element
test_X = np.mean(test_X_tta_list, axis=0)

# Apply L2 normalization again (unit vector length for SVM/kNN is 1)
from sklearn.preprocessing import normalize
test_X = normalize(test_X, norm='l2', axis=1)

print("✅ TTA Feature fusion complete!")

# ==========================================
# 4. Fit lightweight classifiers (five-fold cross-validation)
# ==========================================
print("🧠 Training logistic regression and kNN classifiers...")
n_splits = 5
seed = 42
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

lr_test_preds = np.zeros(len(test_X))
knn_test_preds = np.zeros(len(test_X))

for fold, (train_idx, val_idx) in enumerate(skf.split(train_X, train_y)):
    print(f"--- Fold {fold + 1}/{n_splits} ---")
    X_tr, y_tr = train_X[train_idx], train_y[train_idx]
    
    # Logistic Regression
    lr_model = make_pipeline(
        StandardScaler(),
        SVC(kernel='rbf', C=1.0, class_weight='balanced', probability=True, random_state=seed + fold)
    )
    lr_model.fit(X_tr, y_tr)
    # With probability=True the model supports predict_proba
    lr_test_preds += lr_model.predict_proba(test_X)[:, 1] / n_splits
    
    # kNN
    knn_model = KNeighborsClassifier(n_neighbors=25, weights='distance', metric='cosine')
    knn_model.fit(X_tr, y_tr)
    knn_test_preds += knn_model.predict_proba(test_X)[:, 1] / n_splits

# ==========================================
# 5. Rank Average and top-85 submission
# ==========================================
print("🏆 Combine the final predictions...")
# Average the two scores
final_scores = (lr_test_preds + knn_test_preds) / 2.0

# Extract bare filenames
clean_test_ids = [Path(p).name for p in test_ids]

sub_df = pd.DataFrame({'id': clean_test_ids, 'prob': final_scores})
sub_df = sub_df.sort_values(by='prob', ascending=False).reset_index(drop=True)

# Mark the 85 highest scores positive (meteorites)
top_k = 85
sub_df['label'] = 0
sub_df.loc[:top_k - 1, 'label'] = 1

# Align to the competition template and save
template_df = pd.read_csv(CONFIG['data_root'] / 'sample_submission.csv')
template_df['label'] = template_df['id'].map(sub_df.set_index('id')['label'])
template_df['label'] = template_df['label'].astype(int)

output_csv = CONFIG['output_dir'] / 'submission_dinov2_L14_rankavg_top85.csv'
template_df.to_csv(output_csv, index=False)

print(f"✅ Submission saved to: {output_csv}")